In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report


df = pd.read_excel('Fires.xlsx')

In [3]:
df = df.drop(['FOD_ID','FPA_ID','SOURCE_SYSTEM_TYPE','SOURCE_SYSTEM','NWCG_REPORTING_AGENCY','NWCG_REPORTING_UNIT_ID',
       'NWCG_REPORTING_UNIT_NAME', 'SOURCE_REPORTING_UNIT',
       'SOURCE_REPORTING_UNIT_NAME', 'LOCAL_FIRE_REPORT_ID',
       'LOCAL_INCIDENT_ID', 'FIRE_CODE','FIRE_NAME','ICS_209_INCIDENT_NUMBER','ICS_209_NAME','MTBS_ID','MTBS_FIRE_NAME','OWNER_CODE','COMPLEX_NAME','FIPS_CODE', 'FIPS_NAME','Shape'], axis=1)

In [4]:
'''dftimes = df[['OBJECTID','DISCOVERY_DATE', 'DISCOVERY_DOY', 'DISCOVERY_TIME','CONT_DATE', 'CONT_DOY',
       'CONT_TIME']]
dftimes = dftimes.dropna(how='any')
dftimes.head'''

"dftimes = df[['OBJECTID','DISCOVERY_DATE', 'DISCOVERY_DOY', 'DISCOVERY_TIME','CONT_DATE', 'CONT_DOY',\n       'CONT_TIME']]\ndftimes = dftimes.dropna(how='any')\ndftimes.head"

In [5]:
'''from astropy.time import Time


dftimes['DISCOVERY_DATE'] = dftimes['DISCOVERY_DATE'].apply(lambda x: Time(x, format='jd').iso)
dftimes['CONT_DATE'] = dftimes['CONT_DATE'].apply(lambda x: Time(x, format='jd').iso)   
dftimes['DISCOVERY_DATE'] = pd.to_datetime(dftimes['DISCOVERY_DATE']).dt.date
dftimes['CONT_DATE'] = pd.to_datetime(dftimes['CONT_DATE']).dt.date

dftimes['DISCOVERY_DATE'] = dftimes['DISCOVERY_DATE'].astype(str)
dftimes['CONT_DATE'] = dftimes['CONT_DATE'].astype(str)
dftimes['DISCOVERY_TIME'] = dftimes['DISCOVERY_TIME'].apply(lambda t: f"{int(t)//100:02d}:{int(t)%100:02d}:00")
dftimes['CONT_TIME'] = dftimes['CONT_TIME'].apply(lambda t: f"{int(t)//100:02d}:{int(t)%100:02d}:00")

# Combine date + time
dftimes['DISCOVERY_DATETIME'] = pd.to_datetime(dftimes['DISCOVERY_DATE'] + ' ' + dftimes['DISCOVERY_TIME'])
dftimes['CONT_DATETIME'] = pd.to_datetime(dftimes['CONT_DATE'] + ' ' + dftimes['CONT_TIME'])
dftimes.head()


'''

'from astropy.time import Time\n\n\ndftimes[\'DISCOVERY_DATE\'] = dftimes[\'DISCOVERY_DATE\'].apply(lambda x: Time(x, format=\'jd\').iso)\ndftimes[\'CONT_DATE\'] = dftimes[\'CONT_DATE\'].apply(lambda x: Time(x, format=\'jd\').iso)   \ndftimes[\'DISCOVERY_DATE\'] = pd.to_datetime(dftimes[\'DISCOVERY_DATE\']).dt.date\ndftimes[\'CONT_DATE\'] = pd.to_datetime(dftimes[\'CONT_DATE\']).dt.date\n\ndftimes[\'DISCOVERY_DATE\'] = dftimes[\'DISCOVERY_DATE\'].astype(str)\ndftimes[\'CONT_DATE\'] = dftimes[\'CONT_DATE\'].astype(str)\ndftimes[\'DISCOVERY_TIME\'] = dftimes[\'DISCOVERY_TIME\'].apply(lambda t: f"{int(t)//100:02d}:{int(t)%100:02d}:00")\ndftimes[\'CONT_TIME\'] = dftimes[\'CONT_TIME\'].apply(lambda t: f"{int(t)//100:02d}:{int(t)%100:02d}:00")\n\n# Combine date + time\ndftimes[\'DISCOVERY_DATETIME\'] = pd.to_datetime(dftimes[\'DISCOVERY_DATE\'] + \' \' + dftimes[\'DISCOVERY_TIME\'])\ndftimes[\'CONT_DATETIME\'] = pd.to_datetime(dftimes[\'CONT_DATE\'] + \' \' + dftimes[\'CONT_TIME\'])\ndftimes

In [6]:
'''df = df.merge(
    dftimes[['OBJECTID', 'DISCOVERY_DATETIME', 'CONT_DATETIME']],
    on='OBJECTID',
    how='left'   # keeps all rows from df, fills NaN if no match in dftimes
)
df.head()
'''

"df = df.merge(\n    dftimes[['OBJECTID', 'DISCOVERY_DATETIME', 'CONT_DATETIME']],\n    on='OBJECTID',\n    how='left'   # keeps all rows from df, fills NaN if no match in dftimes\n)\ndf.head()\n"

In [7]:
df = df.dropna(subset=['DISCOVERY_DOY','DISCOVERY_TIME', 'LATITUDE', 'LONGITUDE','FIRE_YEAR',
                       'STAT_CAUSE_DESCR', 'OWNER_DESCR', 'STATE'])

df.drop(['OBJECTID','DISCOVERY_DATE','CONT_DATE','CONT_DOY','CONT_TIME','STAT_CAUSE_CODE','FIRE_SIZE','COUNTY'], axis=1, inplace=True)
# df = df.dropna(how='any', inplace=True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 434827 entries, 0 to 836479
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   FIRE_YEAR         434827 non-null  int64  
 1   DISCOVERY_DOY     434827 non-null  int64  
 2   DISCOVERY_TIME    434827 non-null  float64
 3   STAT_CAUSE_DESCR  434827 non-null  object 
 4   FIRE_SIZE_CLASS   434827 non-null  object 
 5   LATITUDE          434827 non-null  float64
 6   LONGITUDE         434827 non-null  float64
 7   OWNER_DESCR       434827 non-null  object 
 8   STATE             434827 non-null  object 
dtypes: float64(3), int64(2), object(4)
memory usage: 33.2+ MB


In [9]:

numeric_features = ['DISCOVERY_DOY',"DISCOVERY_TIME", 'LATITUDE', 'LONGITUDE']
categorical_features = [ 'STAT_CAUSE_DESCR', 'OWNER_DESCR',"STATE"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


In [10]:
# Build the pipeline with preprocessing and a RandomForestClassifier
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=38,class_weight="balanced"))
])


In [11]:
#Define features and target
X = df.drop('FIRE_SIZE_CLASS', axis=1)
y = df['FIRE_SIZE_CLASS']
# Split the dataset into training (80%) and test (20%) sets, stratifying by the target variable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=38, stratify=y)

In [ ]:
# Train the model on the training set
pipeline.fit(X_train, y_train)

In [ ]:
# Use the trained model to predict on the test set
y_pred = pipeline.predict(X_test)


In [ ]:
# Evaluate model performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[33941  6971   368    19    13    16     8]
 [12297 18511  1739    91    37    24    27]
 [ 1923  5367  1532    97    39    25    10]
 [  509   840   258    48    21    14     7]
 [  352   465   154    33    26    24    10]
 [  313   279    70    17    29    20    10]
 [  163   152    39    11    10    15    22]]

Classification Report:
              precision    recall  f1-score   support

           A       0.69      0.82      0.75     41336
           B       0.57      0.57      0.57     32726
           C       0.37      0.17      0.23      8993
           D       0.15      0.03      0.05      1697
           E       0.15      0.02      0.04      1064
           F       0.14      0.03      0.05       738
           G       0.23      0.05      0.09       412

    accuracy                           0.62     86966
   macro avg       0.33      0.24      0.25     86966
weighted avg       0.58      0.62      0.59     86966



In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

clf = BalancedRandomForestClassifier(random_state=38, n_estimators=200)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', clf)
])


NameError: name 'Pipeline' is not defined

In [ ]:
# Train the model on the training set
pipeline.fit(X_train, y_train)

In [ ]:
# Use the trained model to predict on the test set
y_pred = pipeline.predict(X_test)


In [ ]:
# Evaluate model performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
from imblearn.pipeline import Pipeline   # use imblearn's Pipeline!
from imblearn.over_sampling import SMOTE

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=38, class_weight="balanced"))
])


In [ ]:
# Train the model on the training set
pipeline.fit(X_train, y_train)

In [ ]:
# Use the trained model to predict on the test set
y_pred = pipeline.predict(X_test)


In [ ]:
# Evaluate model performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))